# Data loading from drive

In [1]:
#Load data set from the google drive
from google.colab import drive
import pathlib

drive.mount('/content/drive')
!ls '/content/drive/MyDrive/MSC/DataSet/phm/'

Mounted at /content/drive
phm_test.csv  phm_train.csv  phm_train.gsheet


# Common Imports

In [4]:
# Imports
import pandas as pd
import nltk
from nltk.corpus import stopwords
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer  # to encode text to int
from tensorflow.keras.preprocessing.sequence import pad_sequences   # to do padding or truncating
from tensorflow.keras.models import Sequential     # the model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout # layers of the architecture
from tensorflow.keras.callbacks import ModelCheckpoint   # save model
from tensorflow.keras.models import load_model   # load saved model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Bidirectional

import re

# Function definition for all models

In [5]:
# Function load data from drive as csv
def load_from_csv():
    train_data = pd.read_csv('/content/drive/MyDrive/MSC/DataSet/phm/phm_train.csv')
    test_data = pd.read_csv('/content/drive/MyDrive/MSC/DataSet/phm/phm_test.csv')

    print('\nData loaded..')

    return train_data, test_data


# Function for pre process data
def preprocess_dataset(tweet_data):
    x_data = tweet_data['tweet']       # Reviews/Input  --> Object
    y_data = tweet_data['label']    # Sentiment/Output  --> Int

    # PRE-PROCESS REVIEW
    nltk.download('stopwords')
    english_stops = set(stopwords.words('english'))

    x_data = x_data.replace({'<.*?>': ''}, regex = True)          # remove html tag
    x_data = x_data.replace({'[^A-Za-z]': ' '}, regex = True)     # remove non alphabet
    x_data = x_data.apply(lambda tweet: [w for w in tweet.split() if w not in english_stops])  # remove stop words
    x_data = x_data.apply(lambda tweet: [w.lower() for w in tweet])   # lower case

    # y_data already encoded as 0 and 1 ( int values)
    print('\nData pre-process completed..')

    return x_data, y_data

# Function for get taining and testing data
def get_dataset(train_data, test_data):
    # remove tweet_id from df
    train_data = train_data.drop('tweet_id', axis=1)
    test_data = test_data.drop('tweet_id', axis=1)

    train_data = train_data[train_data['label'].isin([0, 1])]
    test_data = test_data[test_data['label'].isin([0, 1])]

    x_train, y_train = preprocess_dataset(train_data)
    x_test, y_test = preprocess_dataset(test_data)

    return x_train, y_train, x_test, y_test

# Function for getting the maximum tweet length
def get_max_length(x_train):
    tweet_length = []
    for tweet in x_train:
        length = len(tweet)
        tweet_length.append(length)
    #print('Max tweet length: ', np.max(tweet_length))
    #print('Min tweet length: ', np.min(tweet_length))
    #print('Mean tweet length:', np.mean(tweet_length))
    return int(np.ceil(np.mean(tweet_length)))

# Function for tokenize data
def tokenize_data(x_train, x_test):
    token = Tokenizer(lower=False)
    #print(x_train[9073]) # Check for 'how can a being be doing xanax' after stop wording it becomes 'xanax'
    token.fit_on_texts(x_train)
    x_train_seq = token.texts_to_sequences(x_train)
    #print(x_train[9073]) # 'xanax' token is 10
    x_test_seq = token.texts_to_sequences(x_test)

    max_length = get_max_length(x_train)

    x_train_pad = pad_sequences(x_train_seq, maxlen=max_length, padding='post', truncating='post')
    x_test_pad = pad_sequences(x_test_seq, maxlen=max_length, padding='post', truncating='post')

    total_words = len(token.word_index) + 1

    print('Maximum tweet length: ', max_length)
    print('Total words: ', total_words)

    return x_train_pad, x_test_pad, max_length, total_words

# Function for test the model
def test_model(x_test,y_test):
    print("\n\nTesting ---------------------------------------------------------------------")
    # Test the model
    y_pred = model.predict(x_test)
    y_pred = np.round(y_pred).astype(int)

    # Get accurately predicted count
    accurate_count = 0
    for i, y in enumerate(y_test):
        if y == y_pred[i]:
            accurate_count += 1

    print('\nCorrect Prediction: {}'.format(accurate_count))
    print('Wrong Prediction: {}'.format(len(y_pred) - accurate_count))
    accuracy = accurate_count/len(y_pred)*100
    print('Accuracy: {}'.format(accuracy))

    return accuracy


# Function for Trains and evaluates a model multiple times to compute average accuracy.
def evaluate_model_repeatedly(model, x_train, y_train, x_test, y_test, n_runs=5, batch_size=128, epochs=5, checkpoint=None):
    accuracies = []

    for run in range(n_runs):
        # Train the model
        print("\n\nTraining ----------------------------------------------------------------")
        model.fit(x_train, y_train,batch_size=batch_size,epochs=epochs,callbacks=[checkpoint] if checkpoint else None)

        # Evaluate and store accuracy
        accuracy = test_model(x_test, y_test)  # Assumes test_model() returns a float
        accuracies.append(accuracy)

    # Compute statistics
    average_accuracy = np.mean(accuracies)
    std_dev = np.std(accuracies)

    return {
        "average_accuracy": average_accuracy,
        "std_dev": std_dev,
        "all_accuracies": accuracies,
    }

def evaluate_model_repeatedly_modified_with_validation(model, x_train, y_train, x_test, y_test, n_runs=5, batch_size=128, epochs=5, checkpoint=None):
    accuracies = []

    for run in range(n_runs):
        print(f"\n\nRun {run+1}/{n_runs} ----------------------------------------------------")

        # Reset model weights
        for layer in model.layers:
            if hasattr(layer, 'kernel_initializer'):
                # Get the initialization operation
                initial_weights = layer.get_weights()
                new_weights = []
                for w in initial_weights:
                    if hasattr(w, 'numpy'):  # For eager tensors
                        new_weights.append(w.numpy())
                    else:
                        new_weights.append(w)
                layer.set_weights(new_weights)

        # Train with validation data (fixes callback warnings)
        model.fit(x_train, y_train,validation_data=(x_test, y_test),  # Enables val_accuracy/val_loss monitoring
            batch_size=batch_size,epochs=epochs,callbacks=checkpoint if checkpoint else None,verbose=1
        )

        # Evaluate and store accuracy
        accuracy = test_model(x_test, y_test)  # Assumes test_model() returns a float
        accuracies.append(accuracy)
        print(f"Run {run+1} Accuracy: {accuracy:.4f}")

    # Compute statistics
    average_accuracy = np.mean(accuracies)
    std_dev = np.std(accuracies)

    return {
        "average_accuracy": average_accuracy,
        "std_dev": std_dev,
        "all_accuracies": accuracies,
    }

# Bidirectional LSTM


In [6]:
# Load data
train_data, test_data = load_from_csv()
# Get preprocessed data
x_train, y_train, x_test, y_test = get_dataset(train_data, test_data)
# Tokenizing data
x_train, x_test, max_length, total_words = tokenize_data(x_train, x_test)

# Build Bi directional LSTM model

# Optimized Hyperparameters
EMBED_DIM = 128  # Increased from 32 for better representation
LSTM_OUT = 96
DROPOUT_RATE = 0.2  # Moderate dropout for regularization
L2_REG = 0.001   # Light L2 regularization
LEARNING_RATE = 0.0005  # Smaller than default Adam LR

# Build Bi-LSTM model with same regularization
model = Sequential()
model.add(Embedding(total_words, EMBED_DIM, input_length=max_length, embeddings_regularizer=l2(L2_REG)))

# Replace LSTM with Bidirectional LSTM
model.add(Bidirectional(LSTM(LSTM_OUT, dropout=DROPOUT_RATE,recurrent_dropout=DROPOUT_RATE/2,kernel_regularizer=l2(L2_REG))))

model.add(Dropout(DROPOUT_RATE))
model.add(Dense(1, activation='sigmoid'))

# same optimizer and compilation
optimizer = Adam(learning_rate=LEARNING_RATE)
model.build(input_shape=(None, max_length))
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])


checkpoint = [
    ModelCheckpoint(
        'models/BiLSTM_optimized.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )
]

print('\nModel Summary ---------------------------------------------------------')
print(model.summary())
print('\n')

n_runs = 5
batch_size = 128
epochs = 5

results = evaluate_model_repeatedly_modified_with_validation(model=model, x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test, n_runs=n_runs, batch_size=batch_size, epochs=epochs, checkpoint=checkpoint)

print('\n\n\nFinal Results -----------------------------------------------------')
average_accuracy_BiLSTM = results["average_accuracy"]
print('\nAverage Accuracy:', average_accuracy_BiLSTM)
print('All Accuracies:', results["all_accuracies"])


Data loaded..


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Data pre-process completed..

Data pre-process completed..
Maximum tweet length:  10
Total words:  12660

Model Summary ---------------------------------------------------------


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

...

 Trainable params: 1,793,473 (6.84 MB)

 Non-trainable params: 0 (0.00 B)

None




Run 1/5 ----------------------------------------------------
Epoch 1/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.7048 - loss: 1.7646 
Epoch 1: val_accuracy improved from -inf to 0.79856, saving model to models/BiLSTM_optimized.h5


79/79 ━━━━━━━━━━━━━━━━━━━━ 20s 122ms/step - accuracy: 0.7050 - loss: 1.7591 - val_accuracy: 0.7986 - val_loss: 0.7042
Epoch 2/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.8147 - loss: 0.6062
Epoch 2: val_accuracy improved from 0.79856 to 0.82168, saving model to models/BiLSTM_optimized.h5


79/79 ━━━━━━━━━━━━━━━━━━━━ 11s 137ms/step - accuracy: 0.8148 - loss: 0.6056 - val_accuracy: 0.8217 - val_loss: 0.5014
Epoch 3/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.8634 - loss: 0.4193
Epoch 3: val_accuracy did not improve from 0.82168
79/79 ━━━━━━━━━━━━━━━━━━━━ 11s 145ms/step - accuracy: 0.8633 - loss: 0.4194 - val_accuracy: 0.8214 - val_loss: 0.4882
Epoch 4/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.8878 - loss: 0.3529
Epoch 4: val_accuracy did not improve from 0.82168
79/79 ━━━━━━━━━━━━━━━━━━━━ 10s 125ms/step - accuracy: 0.8878 - loss: 0.3531 - val_accuracy: 0.8151 - val_loss: 0.5136
Epoch 5/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 0.9102 - loss: 0.3174
Epoch 5: val_accuracy did not improve from 0.82168
79/79 ━━━━━━━━━━━━━━━━━━━━ 9s 114ms/step - accuracy: 0.9101 - loss: 0.3175 - val_accuracy: 0.8166 - val_loss: 0.5755


Testing ---------------------------------------------------------------------
105/105 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step